# Colab task: connectivity check (Redis + HTTP API)

Use this notebook from Colab **before** running the GPU worker to verify:

- **Redis** is reachable (`PING`).
- Optionally **enqueue** one task via the **public HTTP API** (`POST /task` through nginx).

You need network access from Colab to your VPS (firewall/security group allows the ports you expose).

## 1) Install minimal deps

In [ ]:
%pip install -q redis requests

## 2) Redis ping

In [ ]:
import os
from getpass import getpass

import redis

url = os.environ.get("REDIS_URL") or getpass("REDIS_URL: ")
r = redis.from_url(url, decode_responses=True)
print("PING:", r.ping())

## 3) Submit a smoke task (requires API reachable from Colab)

Set **`BASE_URL`** to your nginx or FastAPI URL, e.g. `http://YOUR_VPS_IP` or HTTPS tunnel endpoint.

In [ ]:
import os
import time

import requests

BASE_URL = os.environ.get("BASE_URL") or "http://YOUR_VPS_IP:8000"
BASE_URL = BASE_URL.rstrip("/")

r = requests.post(f"{BASE_URL}/task", json={"query": "Smoke test from Colab."}, timeout=30)
r.raise_for_status()
task_id = r.json()["task_id"]
print("task_id:", task_id)

for i in range(60):
    g = requests.get(f"{BASE_URL}/task/{task_id}", timeout=30)
    g.raise_for_status()
    body = g.json()
    status = body.get("status")
    print(i, "status:", status)
    if status in ("done", "error"):
        print(body)
        break
    time.sleep(2)
else:
    raise TimeoutError("Still not done — is a worker running and connected to the same Redis?")